# Single signal analysis

This notebook performs a statistical analysis of individual seismic acceleration signals.
For each signal, the analysis covers: empirical probability density functions, 
Gaussian fit and normality assessment, heavy-tail characterization, 
moment scaling analysis, and displacement autocorrelation functions.

## 1. Imports and visualization settings

In [ ]:
from pathlib import Path
import pandas as pd
import os
from src.signals import (plot_empirical_pdfs, gaussian_fit_analysis, heavy_tail_assessment, compute_moment_scaling_acc,
                         compute_moment_scaling_vel, compute_moment_scaling_disp, compute_scaling_exponents,
                         test_scaling_linearity, fit_piecewise_scaling, compute_displacement_autocorrelation,
                         analyze_autocorrelation_scaling)
from src.plot_settings import set_plot_style
from src.latex_export import heavy_tail_to_latex
colors = set_plot_style()

## 2. Data loading

Preprocessed acceleration data are loaded from the parquet file produced 
in notebook 02. Each signal has been baseline-corrected and normalized 
by its standard deviation, so that $\bar{a} = 0$ and $\sigma_a = 1$.

In [ ]:
# Load preprocessed data
df_acc_clean = pd.read_parquet('../data/processed/acc_preprocessed_all.parquet')
print("Shape:", df_acc_clean.shape)
display(df_acc_clean.head())

Shape: (2614815, 4)


,file,sample,acceleration,acceleration_normalized
0,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,0,-6.666667e-10,-3.401661e-08
1,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,1,-6.666667e-10,-3.401661e-08
2,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,2,-6.666667e-10,-3.401661e-08
3,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,3,-6.666667e-10,-3.401661e-08
4,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,4,-6.666667e-10,-3.401661e-08


In [3]:
# Figures output directory
FIGURES_DIR = Path('../figures/03_single_signal')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 3. PDF analysis

Empirical probability density functions are computed for each signal 
using a histogram estimator. Signals are analyzed in normalized units 
to allow comparison across stations.

### 3.1 Empirical PDF per signal

One PDF plot is produced for each of the 66 signals and saved in the 
figures directory.

In [4]:
# Plot empirical PDFsfor each signal
plot_empirical_pdfs(df_acc_clean, bins=100, log_scale=True, output_dir=FIGURES_DIR / 'pdf_single')

Saved: 66/66 PDFs
All PDFs saved successfully!


### 3.2 Gaussian fit and normality assessment

A Gaussian distribution $\mathcal{N}(\mu, \sigma^2)$ is fitted to each 
signal using maximum likelihood estimation. The Anderson-Darling test 
statistic is defined as:

$$A^2 = -n - \frac{1}{n} \sum_{i=1}^{n} (2i-1) \left[ \ln F(x_i) + \ln(1 - F(x_{n+1-i})) \right]$$

where $F$ is the cumulative distribution function of the fitted Gaussian 
and $x_1 \leq \ldots \leq x_n$ are the ordered observations. 
The null hypothesis of normality is rejected at significance level 
$\alpha = 0.05$ if $A^2$ exceeds the critical value.

Kurtosis and skewness are computed as quantitative measures of the 
shape of the distribution. For a Gaussian distribution, the excess 
kurtosis (Fisher's definition) is $\kappa = 0$ and the skewness is $\gamma = 0$:

$$\kappa = \frac{\mu_4}{\sigma^4} - 3, \qquad \gamma = \frac{\mu_3}{\sigma^3}$$

where $\mu_k = \mathbb{E}[(a - \bar{a})^k]$ denotes the $k$-th central moment.

A Q-Q plot against the Gaussian distribution is produced for each signal 
to visualize deviations from normality.

In [5]:
# Run Gaussian fit analysis
df_gaussian_results = gaussian_fit_analysis(df_acc_clean, bins=100, log_scale=True, output_dir=FIGURES_DIR / 'gaussian_fit')

Saved: 66/66 individual plots
All individual plots saved successfully!
Non-Gaussian signals (AD test, p<5%): 66/66


In [6]:
# Display Gaussian fit results
pd.set_option('display.max_rows', None)
display(df_gaussian_results[['station', 'stream', 'kurtosis', 'skewness', 'ad_statistic', 'ad_critical_5pct', 'non_gaussian']])
pd.reset_option('display.max_rows')

,station,stream,kurtosis,skewness,ad_statistic,ad_critical_5pct,non_gaussian
0,EILF,HNE,52.7345,0.4400,6406.5000,0.787,True
1,EILF,HNN,49.1132,0.0666,7521.0485,0.787,True
2,EILF,HNZ,32.1349,-0.2516,5393.3833,0.787,True
3,ESCA,HNE,89.7215,0.1864,9941.4952,0.787,True
4,ESCA,HNN,52.5049,-0.4257,9786.6579,0.787,True
5,ESCA,HNZ,48.1909,-0.6925,9816.3925,0.787,True
6,ISO,HNE,92.0075,0.3071,12600.6412,0.787,True
7,ISO,HNN,93.9731,0.8908,12227.7182,0.787,True
8,ISO,HNZ,158.7818,-0.6685,12948.5878,0.787,True
9,MFC,HNE,30.2292,0.2099,9029.6258,0.787,True


In [7]:
# Save Gaussian fit results
try:
    df_gaussian_results.to_parquet('../data/processed/gaussian_fit_results.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


### 3.3 Heavy-tail assessment

Three alternative distributions are fitted to each signal using maximum 
likelihood estimation:

- **Laplace**: $f(a) = \frac{1}{2b} \exp\left(-\frac{|a - \mu|}{b}\right)$, 
  with exponential tails
- **Student-t**: $f(a) \propto \left(1 + \frac{(a-\mu)^2}{\nu \sigma^2}\right)^{-(\nu+1)/2}$, 
  with power law tails controlled by the degrees of freedom $\nu$
- **Lévy stable**: characterized by the stability index $\alpha \in (0, 2]$ 
  and the skewness parameter $\beta \in [-1, 1]$

The best fit is selected using the Akaike Information Criterion:

$$\text{AIC} = 2k - 2\ln\hat{L}$$

where $k$ is the number of parameters and $\hat{L}$ is the maximized likelihood.

The complementary cumulative distribution function (CCDF) 
$P(|a| > x)$ is plotted on a log-log scale. If the tail follows a 
power law $P(|a| > x) \sim x^{-\alpha}$, it appears as a straight line 
with slope $-\alpha$. The power law exponent is estimated using the 
Hill estimator:

$$\hat{\alpha} = \left( \frac{1}{k} \sum_{i=1}^{k} \ln \frac{x_{(i)}}{x_{(k+1)}} \right)^{-1}$$

where $x_{(1)} \geq \ldots \geq x_{(n)}$ are the ordered observations 
and $k$ is the number of tail observations (top 10\% of values).

In [5]:
# Check partial results if available
try:
    df_partial = pd.read_parquet('../data/processed/heavy_tail_results_partial.parquet')
    print(f"Partial results available: {len(df_partial)} signals already processed")
except Exception:
    print("No partial results found, starting from scratch")

Partial results available: 66 signals already processed


In [6]:
# Run heavy-tail assessment
df_heavy_tail_results = heavy_tail_assessment(df_acc_clean, output_dir=FIGURES_DIR / 'heavy_tail', resume=True )

Resuming from 66 already processed signals
Saved: 0/66 individual plots
All individual plots saved successfully!

Best fit by AIC:
best_fit_aic
Levy-stable    55
Student-t      11
Name: count, dtype: int64


In [7]:
# Display heavy-tail assessment results
pd.set_option('display.max_rows', None)
display(df_heavy_tail_results[['station', 'stream', 'aic_gaussian', 'aic_laplace', 
                                'aic_student_t', 'aic_levy_stable', 'best_fit_aic', 'student_t_df', 
                                'power_law_exp']])
pd.reset_option('display.max_rows')

,station,stream,aic_gaussian,aic_laplace,aic_student_t,aic_levy_stable,best_fit_aic,student_t_df,power_law_exp
0,EILF,HNE,136221.10,75859.86,52804.13,52746.79,Levy-stable,1.2651,1.1420
1,EILF,HNN,136221.10,68603.98,33000.83,33018.55,Student-t,0.9964,1.1200
2,EILF,HNZ,136221.10,85179.23,67682.48,67581.97,Levy-stable,1.3937,1.1692
3,ESCA,HNE,136221.10,46302.74,-41994.24,-42357.24,Levy-stable,0.6018,1.0875
4,ESCA,HNN,136221.10,51314.78,-35903.50,-36218.17,Levy-stable,0.6116,1.0638
5,ESCA,HNZ,136221.10,52175.36,-28865.09,-28829.31,Student-t,0.6807,1.0188
6,ISO,HNE,136221.10,19666.20,-100500.55,-100096.19,Student-t,0.7025,0.7322
7,ISO,HNN,136221.10,23524.52,-88695.53,-88301.55,Student-t,0.7135,0.7902
8,ISO,HNZ,136221.10,14731.63,-112766.34,-112240.15,Student-t,0.7140,0.6845
9,MFC,HNE,136221.10,60178.50,-27107.33,-28590.05,Levy-stable,0.5136,1.1546


In [8]:
# Save results
try:
    df_heavy_tail_results.to_parquet('../data/processed/heavy_tail_results.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


In [8]:
# Generate LaTeX table for the heavy-tail assessment appendix
latex_rows = heavy_tail_to_latex(
    df_heavy_tail_results,
    output_path='../data/processed/heavy_tail_table.tex'
)

# Preview the first few lines to verify formatting
print(latex_rows[:500])

Saved to: ../data/processed/heavy_tail_table.tex
EILF & HNE & 52,746.79 & 52,804.13 & L\'evy-stable & 1.2651 & 1.1420 \\
EILF & HNN & 33,018.55 & 33,000.83 & Student-$t$ & 0.9964 & 1.1200 \\
EILF & HNZ & 67,581.97 & 67,682.48 & L\'evy-stable & 1.3937 & 1.1692 \\
\addlinespace
ESCA & HNE & $-$42,357.24 & $-$41,994.24 & L\'evy-stable & 0.6018 & 1.0875 \\
ESCA & HNN & $-$36,218.17 & $-$35,903.50 & L\'evy-stable & 0.6116 & 1.0638 \\
ESCA & HNZ & $-$28,829.31 & $-$28,865.09 & Student-$t$ & 0.6807 & 1.0188 \\
\addlinespace
ISO & HNE & $-$100,096.19 


## 4. Moment scaling analysis - acceleration

### 4.1 Computation of q-th order moments

Moments are computed for a range of moment orders q and time scales tau. 
The resulting dataframe contains one row per $(\text{file}, q, \tau)$ combination.

In [4]:
q_values = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
tau_values = [10, 50, 100, 200, 500, 1000, 2000, 5000, 10000]

# Stations excluded from moment scaling due to short signal length
short_stations = ['SURF', 'BRZ', 'BHB', 'CRI', 'SLZ', 'SAV']

df_acc_scaling = df_acc_clean[
    ~df_acc_clean['file'].str.split('.').str[1].isin(short_stations)
].copy()

print(f"Signals used for moment scaling: {df_acc_scaling['file'].nunique()} / {df_acc_clean['file'].nunique()}")

Signals used for moment scaling: 48 / 66


In [5]:
df_moments_acc = compute_moment_scaling_acc(df_acc_scaling, q_values, tau_values)
print(df_moments_acc.shape)
display(df_moments_acc)

(3456, 6)


,file,station,stream,q,tau,moment
0,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,0.5,10,0.591691
1,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,1.0,10,0.587259
2,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,1.5,10,0.938694
3,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,2.0,10,2.129010
4,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,2.5,10,5.949601
...,...,...,...,...,...,...
3451,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,2.0,10000,2.524536
3452,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,2.5,10000,5.608439
3453,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,3.0,10000,13.615232
3454,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,3.5,10000,35.341492


### 4.2 Scaling exponents estimation

For each signal and each moment order $q$, the scaling exponent $\zeta(q)$ 
is estimated by linear regression of $\log M_q(\tau)$ vs $\log \tau$:

$$\log M_q(\tau) = \zeta(q) \log \tau + c$$

The slope $\zeta(q)$ is the scaling exponent. A plot of $\zeta(q)$ vs $q$ 
is produced for each signal, with the reference line $\zeta(q) = q/2$ 
corresponding to normal diffusion.

In [ ]:
df_exponents = compute_scaling_exponents(df_moments_acc, output_dir=FIGURES_DIR / 'scaling'/ 'acceleration' / 'exponents')

Saved: 48/48 individual plots
All individual plots saved successfully!


In [ ]:
# Save results
try:
    df_exponents.to_parquet('../data/processed/scaling_exponents_acc.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


### 4.3 Linearity check

The linearity of $\zeta(q)$ vs $q$ is assessed by comparing two models 
using AIC:

- **Linear**: $\zeta(q) = aq + b$
- **Quadratic**: $\zeta(q) = aq^2 + bq + c$

If the quadratic model is preferred, the scaling is non-linear, 
indicating anomalous diffusion.

A piecewise linear fit is also performed to detect the presence of 
two distinct scaling regimes, consistent with the framework of strong 
anomalous diffusion (Vollmer et al., 2021):

$$\zeta(q) = \begin{cases} \phi q & q \leq q^* \\ \lambda q - q^*(\lambda - \phi) & q > q^* \end{cases}$$

where $q^*$ is the breakpoint and $\phi$, $\lambda$ are the slopes of 
the two regimes. The breakpoint $q^*$ is estimated by minimizing the 
total residual sum of squares.

In [ ]:
# Linearity check
df_linearity = test_scaling_linearity(df_exponents, output_dir=FIGURES_DIR / 'scaling'/ 'acceleration' / 'linearity')

Saved: 48/48 individual plots
All individual plots saved successfully!

Best fit by AIC:
best_fit
quadratic    46
linear        2
Name: count, dtype: int64


In [ ]:
# Save results
try:
    df_linearity.to_parquet('../data/processed/scaling_linearity_acc.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


In [ ]:
# Piecewise linear fit
df_piecewise = fit_piecewise_scaling(df_exponents, output_dir=FIGURES_DIR / 'scaling'/ 'acceleration' / 'piecewise')

Saved: 48/48 individual plots
All individual plots saved successfully!


In [ ]:
# Save results
try:
    df_piecewise.to_parquet('../data/processed/scaling_piecewise_acc.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


## 5. Moment scaling analysis - velocity

### 5.1 Computation of q-th order moments

Moments are computed for a range of moment orders q and time scales tau. 
The resulting dataframe contains one row per $(\text{file}, q, \tau)$ combination.

In [ ]:
q_values = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
tau_values = [10, 50, 100, 200, 500, 1000, 2000, 5000, 10000]

# Stations excluded from moment scaling due to short signal length
short_stations = ['SURF', 'BRZ', 'BHB', 'CRI', 'SLZ', 'SAV']

df_vel_scaling = df_acc_clean[
    ~df_acc_clean['file'].str.split('.').str[1].isin(short_stations)
].copy()

print(f"Signals used for moment scaling: {df_vel_scaling['file'].nunique()} / {df_acc_clean['file'].nunique()}")

In [ ]:
df_moments_vel = compute_moment_scaling_vel(df_vel_scaling, q_values, tau_values)
print(df_moments_vel.shape)
display(df_moments_vel)

### 5.2 Scaling exponents estimation

For each signal and each moment order $q$, the scaling exponent $\zeta(q)$ 
is estimated by linear regression of $\log M_q(\tau)$ vs $\log \tau$:

$$\log M_q(\tau) = \zeta(q) \log \tau + c$$

The slope $\zeta(q)$ is the scaling exponent. A plot of $\zeta(q)$ vs $q$ 
is produced for each signal, with the reference line $\zeta(q) = q/2$ 
corresponding to normal diffusion.

In [ ]:
df_exponents = compute_scaling_exponents(df_moments_vel, output_dir=FIGURES_DIR / 'scaling'/ 'velocity' / 'exponents')

In [ ]:
# Save results
try:
    df_exponents.to_parquet('../data/processed/scaling_exponents_vel.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

### 5.3 Linearity check

The linearity of $\zeta(q)$ vs $q$ is assessed by comparing two models 
using AIC:

- **Linear**: $\zeta(q) = aq + b$
- **Quadratic**: $\zeta(q) = aq^2 + bq + c$

If the quadratic model is preferred, the scaling is non-linear, 
indicating anomalous diffusion.

A piecewise linear fit is also performed to detect the presence of 
two distinct scaling regimes, consistent with the framework of strong 
anomalous diffusion (Vollmer et al., 2021):

$$\zeta(q) = \begin{cases} \phi q & q \leq q^* \\ \lambda q - q^*(\lambda - \phi) & q > q^* \end{cases}$$

where $q^*$ is the breakpoint and $\phi$, $\lambda$ are the slopes of 
the two regimes. The breakpoint $q^*$ is estimated by minimizing the 
total residual sum of squares.

In [ ]:
# Linearity check
df_linearity = test_scaling_linearity(df_exponents, output_dir=FIGURES_DIR / 'scaling'/ 'velocity' / 'linearity')

In [ ]:
# Save results
try:
    df_piecewise.to_parquet('../data/processed/scaling_piecewise_vel.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

## 6. Moment scaling analysis - 

### 6.1 Computation of q-th order moments

Moments are computed for a range of moment orders q and time scales tau. 
The resulting dataframe contains one row per $(\text{file}, q, \tau)$ combination.

In [ ]:
q_values = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
tau_values = [10, 50, 100, 200, 500, 1000, 2000, 5000, 10000]

# Stations excluded from moment scaling due to short signal length
short_stations = ['SURF', 'BRZ', 'BHB', 'CRI', 'SLZ', 'SAV']

df_disp_scaling = df_acc_clean[
    ~df_acc_clean['file'].str.split('.').str[1].isin(short_stations)
].copy()

print(f"Signals used for moment scaling: {df_disp_scaling['file'].nunique()} / {df_acc_clean['file'].nunique()}")

In [ ]:
df_moments_disp = compute_moment_scaling_disp(df_disp_scaling, q_values, tau_values)
print(df_moments_disp.shape)
display(df_moments_disp)

### 6.2 Scaling exponents estimation

For each signal and each moment order $q$, the scaling exponent $\zeta(q)$ 
is estimated by linear regression of $\log M_q(\tau)$ vs $\log \tau$:

$$\log M_q(\tau) = \zeta(q) \log \tau + c$$

The slope $\zeta(q)$ is the scaling exponent. A plot of $\zeta(q)$ vs $q$ 
is produced for each signal, with the reference line $\zeta(q) = q/2$ 
corresponding to normal diffusion.

In [ ]:
df_exponents = compute_scaling_exponents(df_moments_disp, output_dir=FIGURES_DIR / 'scaling'/ 'displacement' / 'exponents')

In [ ]:
# Save results
try:
    df_exponents.to_parquet('../data/processed/scaling_exponents_disp.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

### 6.3 Linearity check

The linearity of $\zeta(q)$ vs $q$ is assessed by comparing two models 
using AIC:

- **Linear**: $\zeta(q) = aq + b$
- **Quadratic**: $\zeta(q) = aq^2 + bq + c$

If the quadratic model is preferred, the scaling is non-linear, 
indicating anomalous diffusion.

A piecewise linear fit is also performed to detect the presence of 
two distinct scaling regimes, consistent with the framework of strong 
anomalous diffusion (Vollmer et al., 2021):

$$\zeta(q) = \begin{cases} \phi q & q \leq q^* \\ \lambda q - q^*(\lambda - \phi) & q > q^* \end{cases}$$

where $q^*$ is the breakpoint and $\phi$, $\lambda$ are the slopes of 
the two regimes. The breakpoint $q^*$ is estimated by minimizing the 
total residual sum of squares.

In [ ]:
# Linearity check
df_linearity = test_scaling_linearity(df_exponents, output_dir=FIGURES_DIR / 'scaling'/ 'displacement' / 'linearity')

In [ ]:
# Save results
try:
    df_piecewise.to_parquet('../data/processed/scaling_piecewise_disp.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

## 5. Displacement autocorrelation functions

The displacement autocorrelation function is computed for each signal 
following the framework of Vollmer et al. (2021):

$$C(t_1, t_2) = \langle (a(t_1) - a_0)(a(t_2) - a_0) \rangle$$

where $a_0 = a(0)$ is the initial value of the signal. The function 
is evaluated on a logarithmic grid of $(t_1, t_2)$ pairs to capture 
scaling behavior across multiple time scales.

### 5.1 Computation

Autocorrelation functions are computed on a logarithmic grid of $n_{points}$ values for both $t_1$ and $t_2$, resulting in a matrix of size $n_{points} \times n_{points}$ for each signal.

In [ ]:
df_autocorr, C_matrices = compute_displacement_autocorrelation(df_acc_clean, output_dir=FIGURES_DIR / 'autocorrelation')

### 5.2 Scaling behavior

The scaling exponent $\beta$ is estimated from the diagonal $C(t, t)$, 
which is expected to scale as:

$$C(t, t) \sim t^\beta$$

The exponent $\beta$ is estimated by linear regression of 
$\log |C(t, t)|$ vs $\log t$. A value of $\beta = 1$ is consistent 
with normal diffusion, while $\beta \neq 1$ indicates anomalous scaling.

In [ ]:
df_autocorr_scaling = analyze_autocorrelation_scaling(
    C_matrices, output_dir=FIGURES_DIR / 'autocorrelation')

In [ ]:
# Save results
try:
    df_autocorr_scaling.to_parquet('../data/processed/autocorr_scaling.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

## 6. Summary

A summary table collects the main results from all analyses for each signal:

| Column | Description |
|--------|-------------|
| `kurtosis` | Excess kurtosis $\kappa$ |
| `skewness` | Skewness $\gamma$ |
| `non_gaussian` | Anderson-Darling test result ($\alpha = 0.05$) |
| `best_fit_aic` | Best fitting distribution by AIC |
| `student_t_df` | Student-t degrees of freedom $\nu$ |
| `power_law_exp` | Power law exponent $\hat{\alpha}$ (Hill estimator) |
| `q_break` | Piecewise scaling breakpoint $q^*$ |
| `slope_low` | Scaling slope for $q \leq q^*$ |
| `slope_high` | Scaling slope for $q > q^*$ |
| `beta` | Autocorrelation scaling exponent $\beta$ |

In [ ]:
df_gaussian_results  = pd.read_parquet('../data/processed/gaussian_fit_results.parquet')
df_heavy_tail_results = pd.read_parquet('../data/processed/heavy_tail_results.parquet')
df_piecewise         = pd.read_parquet('../data/processed/scaling_piecewise.parquet')
df_autocorr_scaling  = pd.read_parquet('../data/processed/autocorr_scaling.parquet')

df_summary = df_gaussian_results[['station', 'stream', 'kurtosis', 'skewness', 'non_gaussian']]\
    .merge(df_heavy_tail_results[['station', 'stream', 'best_fit_aic', 'student_t_df', 'power_law_exp']],
           on=['station', 'stream'])\
    .merge(df_piecewise[['station', 'stream', 'q_break', 'slope_low', 'slope_high']],
           on=['station', 'stream'])\
    .merge(df_autocorr_scaling[['station', 'stream', 'beta', 'r2']],
           on=['station', 'stream'])

display(df_summary)

try:
    df_summary.to_parquet('../data/processed/summary.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")